## `Main code`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [2]:
sns.set_theme(style="whitegrid")

In [3]:
df = pd.read_csv('../data/main_df.csv')

In [4]:
display(df.head())

,food_code,food_name,primarysource,energy_kj,energy_kcal,carb_g,protein_g,fat_g,freesugar_g,fibre_g,...,unit_serving_vitb2_mg,unit_serving_vitb3_mg,unit_serving_vitb5_mg,unit_serving_vitb6_mg,unit_serving_vitb7_ug,unit_serving_vitb9_ug,unit_serving_vitc_mg,unit_serving_carotenoids_ug,Course_Type,Diet_Type
0,ASC001,Hot tea (Garam Chai),asc_manual,68.16,16.14,2.58,0.39,0.53,2.58,0.00,...,0.03,0.02,0.09,0.01,0.51,1.80,0.50,50.00,Beverage/Side,Veg
1,ASC002,Instant coffee,asc_manual,97.77,23.16,3.65,0.64,0.75,3.62,0.00,...,0.09,0.80,0.25,0.03,3.49,5.60,1.51,150.00,Beverage/Side,Veg
2,ASC003,Espreso coffee,asc_manual,217.00,51.54,6.62,1.75,2.14,6.53,0.00,...,0.09,0.56,0.26,0.03,2.87,5.53,1.51,150.00,Beverage/Side,Veg
3,ASC004,Iced tea,asc_manual,44.09,10.34,2.70,0.03,0.01,2.70,0.00,...,0.00,0.02,0.02,0.01,0.02,1.28,5.95,0.70,Beverage/Side,Veg
4,ASC005,Raw mango drink (Aam panna),asc_manual,152.39,35.92,9.05,0.16,0.03,7.49,0.61,...,0.01,0.14,0.07,0.07,0.73,14.05,45.30,448.89,Beverage/Side,Vegan


In [5]:
print("\n--- Missing Values Check ---")
print(df.isnull().sum()) 


--- Missing Values Check ---
food_code                       0
food_name                       0
primarysource                   0
energy_kj                       0
energy_kcal                     0
                               ..
unit_serving_vitb9_ug          77
unit_serving_vitc_mg           77
unit_serving_carotenoids_ug    77
Course_Type                     0
Diet_Type                       0
Length: 84, dtype: int64


In [6]:
critical_cols = [
    'unit_serving_fibre_g', 'unit_serving_potassium_mg', 'unit_serving_magnesium_mg', 
    'unit_serving_calcium_mg', 'unit_serving_protein_g', 'unit_serving_sodium_mg', 
    'unit_serving_sfa_mg', 'unit_serving_freesugar_g'
]

print(f"Total dishes before cleaning: {len(df)}")

df.dropna(subset=critical_cols, inplace=True)

print(f"Total dishes after dropping invalid rows: {len(df)}")


Total dishes before cleaning: 999
Total dishes after dropping invalid rows: 922


In [7]:
# --- PHASE 1: The Hard Filter ---
# Eliminate any single serving with more than 800mg of sodium
unsafe_threshold = 800
safe_df = df[df['unit_serving_sodium_mg'] <= unsafe_threshold].copy()

print(f"Safe, clinically valid meals for the engine: {len(safe_df)}")

Safe, clinically valid meals for the engine: 847


## `PHASE 2: THE DASH SCORER`


In [8]:
metrics_to_scale = [
    'unit_serving_fibre_g', 'unit_serving_potassium_mg', 'unit_serving_magnesium_mg', 
    'unit_serving_calcium_mg', 'unit_serving_protein_g', 'unit_serving_sodium_mg', 
    'unit_serving_sfa_mg', 'unit_serving_freesugar_g'
]

# Apply Z-Score Standardization
scaler = StandardScaler()
safe_df_scaled = safe_df.copy()
safe_df_scaled[metrics_to_scale] = scaler.fit_transform(safe_df[metrics_to_scale])

def calculate_zscore_dash(row):
    """
    Weights approximate the combined protective effect of High K + High Mg + Adequate Ca against Na.
    """
    # REWARDS (Cardiovascular Promoters)
    fiber_pts = row['unit_serving_fibre_g'] * 5.0          
    potassium_pts = row['unit_serving_potassium_mg'] * 3.5 
    
    magnesium_pts = row['unit_serving_magnesium_mg'] * 2.0 
    calcium_pts = row['unit_serving_calcium_mg'] * 2.0     
    protein_pts = row['unit_serving_protein_g'] * 1.0      
    
    # PENALTIES (Hypertension Exacerbators)
    sodium_pen = row['unit_serving_sodium_mg'] * 4.0       
    sat_fat_pen = row['unit_serving_sfa_mg'] * 3.0         
    sugar_pen = row['unit_serving_freesugar_g'] * 2.0      
    
    rewards = fiber_pts + potassium_pts + magnesium_pts + calcium_pts + protein_pts
    penalties = sodium_pen + sat_fat_pen + sugar_pen
    
    return round((rewards - penalties), 3)

# Apply the score and rank
safe_df['DASH_Score'] = safe_df_scaled.apply(calculate_zscore_dash, axis=1)
ranked_meals = safe_df.sort_values(by='DASH_Score', ascending=False)

# Display the top 5 results
display_cols = [
    'food_name', 'DASH_Score', 'unit_serving_potassium_mg', 
    'unit_serving_sodium_mg', 'unit_serving_fibre_g', 'unit_serving_sfa_mg'
]

print("\n--- TOP 5 CLINICALLY OPTIMAL INDIAN MEALS (K-Adjusted) ---")
display(ranked_meals[display_cols].head(5))


--- TOP 5 CLINICALLY OPTIMAL INDIAN MEALS (K-Adjusted) ---


,food_name,DASH_Score,unit_serving_potassium_mg,unit_serving_sodium_mg,unit_serving_fibre_g,unit_serving_sfa_mg
210,Spinach mutton (Palak mutton),52.985,2256.47,588.57,10.85,1927.08
511,Lasagne with vegetables,45.431,2221.88,716.66,11.66,18624.06
795,Lotus seed halwa (Kamal gattay ka halwa),42.425,2387.13,11.08,1.53,40028.29
793,Kiwi granola pudding,41.998,1438.54,310.28,19.44,29701.95
201,Avial,37.105,1267.49,273.05,18.24,20865.47


## `PHASE 3: METABOLIC ENGINE & PLAN GENERATOR`


In [ ]:
def calculate_dash_tdee_and_constraints(weight_kg, height_cm, age_years, gender, activity_level, bp_stage):
    """
    Standard Mifflin-St Jeor with Indian Metabolic Adjustment (-5%) 
    and Stage-specific Sodium Tiers.
    """
    if gender.lower() == 'male':
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years + 5
    else:  
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years - 161
        
    multipliers = {'sedentary': 1.2, 'light': 1.375, 'moderate': 1.55, 'active': 1.725}
    indian_tdee = (bmr * multipliers.get(activity_level, 1.2)) * 0.95
    
    sodium_daily = {'pre': 2300, 'stage1': 2000, 'stage2': 1500}
    dash_limit = sodium_daily[bp_stage]
    
    # Target Calories per slot
    meals = {
        'Breakfast': round(indian_tdee * 0.25),
        'Lunch': round(indian_tdee * 0.30),
        'Dinner': round(indian_tdee * 0.30),
        'Snack': round(indian_tdee * 0.075) 
    }
    
    # 10% Clinical Sodium Buffer Strategy
    return {
        'indian_tdee': round(indian_tdee),
        'meal_calories': meals,
        'sodium_per_main': round(dash_limit * 0.25),
        'sodium_per_snack': round(dash_limit * 0.075),
        'daily_sodium_limit': dash_limit
    }



def generate_personalized_exchange_plan(ranked_df, constraints, patient_diet_pref='Any', options_per_slot=3, top_n_pool=15):
    """
    Generates a daily plan with DIETARY VARIETY.
    Instead of rigidly picking the top 3, it randomly samples from the Top 15 safest options.
    """
    if patient_diet_pref.lower() != 'any':
        personal_df = ranked_df[ranked_df['Diet_Type'].str.lower() == patient_diet_pref.lower()].copy()
    else:
        personal_df = ranked_df.copy()

    slots = [
        ('Breakfast', constraints['meal_calories']['Breakfast'], constraints['sodium_per_main'], 200, ['Breakfast']),
        ('Lunch', constraints['meal_calories']['Lunch'], constraints['sodium_per_main'], 250, ['Main Course']),
        ('Dinner', constraints['meal_calories']['Dinner'], constraints['sodium_per_main'], 250, ['Main Course']),
        ('Snack 1', constraints['meal_calories']['Snack'], constraints['sodium_per_snack'], 100, ['Snack', 'Beverage/Side']),
        ('Snack 2', constraints['meal_calories']['Snack'], constraints['sodium_per_snack'], 100, ['Snack', 'Beverage/Side'])
    ]
    
    full_exchange_list = []
    used_indices = set() 
    
    for slot_name, target_cal, max_na, tolerance, allowed_courses in slots:
        # 1. Filter for valid courses
        appropriate_foods = personal_df[personal_df['Course_Type'].isin(allowed_courses)]
        
        # 2. Filter for Calorie and Sodium strict constraints (Vectorized for speed)
        valid_pool = appropriate_foods[
            (appropriate_foods['unit_serving_energy_kcal'].between(target_cal - tolerance, target_cal + tolerance)) &
            (appropriate_foods['unit_serving_sodium_mg'] <= max_na) &
            (~appropriate_foods.index.isin(used_indices)) # Exclude already used dishes
        ].copy()
        
        # 3. THE VARIETY UPGRADE: Take the top 'N' best DASH scores, then randomly sample 3
        if not valid_pool.empty:
            # Sort by best DASH score to ensure clinical quality
            top_pool = valid_pool.sort_values(by='DASH_Score', ascending=False).head(top_n_pool)
            
            # Randomly sample 'options_per_slot' (e.g., 3) from this high-quality pool
            sample_size = min(options_per_slot, len(top_pool))
            selected_meals = top_pool.sample(n=sample_size)
            
            selected_meals['Meal_Slot'] = slot_name
            selected_meals['Target_Kcal'] = target_cal
            
            # Add to the global memory tracker
            used_indices.update(selected_meals.index.tolist())
            full_exchange_list.append(selected_meals)
            
    if full_exchange_list:
        return pd.concat(full_exchange_list)
    else:
        return pd.DataFrame()


In [21]:
# 1. Expanded Profile: Rajesh (Stage 2 Hypertension, strictly Vegetarian)
rajesh_profile = {
    'weight_kg': 75, 'height_cm': 165, 'age_years': 55, 
    'gender': 'male', 'activity_level': 'sedentary', 'bp_stage': 'stage2',
    'diet_pref': 'Veg' 
}

# 2. Get Constraints (We pop 'diet_pref' out so it doesn't break the math function)
diet_preference = rajesh_profile.pop('diet_pref')
limits = calculate_dash_tdee_and_constraints(**rajesh_profile)

# 3. Generate Personalized Gallery
exchange_plan = generate_personalized_exchange_plan(ranked_meals, limits, patient_diet_pref=diet_preference)

view_cols = ['Meal_Slot', 'Course_Type', 'Diet_Type', 'food_name', 'DASH_Score', 'unit_serving_energy_kcal']
print(f"--- CLINICAL {diet_preference.upper()} DIET GALLERY FOR RAJESH ({limits['indian_tdee']} kcal) ---")
display(exchange_plan[view_cols].groupby('Meal_Slot').head(3))

--- CLINICAL VEG DIET GALLERY FOR RAJESH (1723 kcal) ---


,Meal_Slot,Course_Type,Diet_Type,food_name,DASH_Score,unit_serving_energy_kcal
484,Breakfast,Breakfast,Veg,Peas poori (Matar ki poori),-2.908,496.79
48,Breakfast,Breakfast,Veg,Cornflakes with milk,5.018,326.17
845,Breakfast,Breakfast,Veg,Corn omelette/omlet,-4.305,324.26
270,Lunch,Main Course,Veg,Semolina halwa (Suji ka halwa),-2.976,360.18
133,Lunch,Main Course,Veg,Onion tomato uttapam,-6.424,644.32
490,Lunch,Main Course,Veg,Eggplant/Brinjal rice (Vangi bhat),32.546,538.09
116,Dinner,Main Course,Veg,Tamarind rice (Chintapandu pulihora/Puliyodhar...,10.431,476.86
847,Dinner,Main Course,Veg,Paaner do pyaza,-3.696,296.11
1,Snack 1,Beverage/Side,Veg,Instant coffee,-3.036,104.20
19,Snack 1,Beverage/Side,Veg,Sweet Lassi (Meethi lassi),-2.012,157.80
